# Preprocessing ERA5-Land district temperature

Merges the per-period ERA5-Land daily temperature extracts into one file,
resolves the duplicated MBALE records, restricts to the study area, and joins the
result onto the flood training frame.

Mirrors `chirps_preprocessing.ipynb`, with one substantive difference in how the
MBALE duplication is resolved — see section 2.

**Input:** `dataset/ERA5_Elgon/era5land_daily_districts_*.csv` — nine files,
`date, district, tmax_c, tmin_c, tmean_c`, daily temperature in °C.

**Outputs:**

| File | Contents |
| --- | --- |
| `dataset/era5_daily_temperature.csv` | Consolidated temperature, 7 districts |
| `dataset/flood_training_data.csv` | Rainfall + temperature + label, model-ready |

| Step | Effect |
| --- | --- |
| 1. Load the nine extracts | 164,360 rows |
| 2. Drop Mbale Municipality | 164,360 → 147,924 |
| 3. Drop BUKWO and KWEEN | 147,924 → 115,052 |
| 4. Join onto the training frame | 48,605 rows, 3 new columns |

## 1. Load the extracts

In [ ]:
from pathlib import Path

import pandas as pd

ERA5_DIR = "dataset/ERA5_Elgon"
CHIRPS_DIR = "dataset/CHIRPS_Elgon"
TEMPERATURE_PATH = "dataset/era5_daily_temperature.csv"
TRAINING_IN = "dataset/chirps_flood_training_data.csv"
TRAINING_OUT = "dataset/flood_training_data.csv"

TEMP_COLUMNS = ["tmax_c", "tmin_c", "tmean_c"]
EXCLUDED_DISTRICTS = ["BUKWO", "KWEEN"]


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()


def load_extracts(directory: str, pattern: str) -> pd.DataFrame:
    """Concatenate per-period extracts, preserving each file's own row order.

    Row order carries the boundary layer's feature order, which is what
    identifies the two MBALE polygons in section 2, so it is recorded explicitly
    rather than left to survive an incidental sort.
    """
    files = sorted((REPO_ROOT / directory).glob(pattern))
    if not files:
        raise FileNotFoundError(f"no extracts matching {pattern} in {REPO_ROOT / directory}")

    frames = []
    for path in files:
        frame = pd.read_csv(path)
        if frame.empty:
            print(f"  skipped {path.name} (header only)")
            continue
        frame["date"] = pd.to_datetime(frame["date"])
        frame["district"] = frame["district"].str.strip().str.upper()
        frame["source_file"] = path.name
        frame["file_row"] = range(len(frame))
        frames.append(frame)
        print(f"  {path.name:<44} {len(frame):>6} rows  "
              f"{frame['date'].min():%Y-%m-%d}..{frame['date'].max():%Y-%m-%d}")
    return pd.concat(frames, ignore_index=True)


temperature = load_extracts(ERA5_DIR, "era5land_daily_districts_*.csv")

print(f"\nloaded {len(temperature):,} rows")
print(f"date range: {temperature['date'].min():%Y-%m-%d} to {temperature['date'].max():%Y-%m-%d}")
print(f"districts ({temperature['district'].nunique()}): "
      f"{', '.join(sorted(temperature['district'].unique()))}")
print(f"nulls: {int(temperature[TEMP_COLUMNS].isna().sum().sum())}")
print(f"tmin <= tmean <= tmax holds: "
      f"{bool(((temperature['tmin_c'] <= temperature['tmean_c']) & (temperature['tmean_c'] <= temperature['tmax_c'])).all())}")
print()
print(temperature.groupby("district")[TEMP_COLUMNS].mean().round(2).sort_values("tmean_c").to_string())

### Findings — coverage

**ERA5-Land runs 1981-01-01 to 2025-12-31 — 16,436 days, 45 years.** That is far
longer than the CHIRPS rainfall record (1998–2025), so **rainfall, not
temperature, is the binding constraint on the study period.** CHIRPS v3 is also
available from 1981; if a longer record is ever wanted, re-extracting CHIRPS back
to 1991 would recover the two pre-1998 flood onsets currently discarded.

The district temperature ordering is physically coherent, which is a useful
sanity check on the extraction: BUKWO (15.8 °C) and BUDUDA (16.5 °C) are coldest,
sitting highest on the massif, and BUTALEJA (22.0 °C) is warmest, being the
lowland district. Temperature tracks elevation as it should.

No nulls, and `tmin <= tmean <= tmax` holds on every row.

## 2. The duplicated MBALE records

As in the CHIRPS extraction, MBALE appears **twice** on every date: the boundary
layer contains both **Mbale Municipality** and the general **Mbale District**, and
the municipality lies inside the district. The district's zonal statistics already
cover the municipality's cells, so the municipality rows are dropped.

### Why the CHIRPS test cannot be reused

For CHIRPS the two polygons were separable by a strict containment argument:
`rain_max` and `rain_min` are *spatial* statistics over grid cells, so the larger
polygon must bracket the smaller one's range.

**That argument does not hold here.** `tmax_c` and `tmin_c` are *diurnal*
aggregates — the day's maximum and minimum temperature — not the spatial extremes
of a single quantity. Their ~8.8 °C separation is the day–night cycle, not
within-district variation. There is no containment guarantee to test.

### The discriminator that does work

Both extractions were run over the same boundary layer, and that layer's feature
order is recoverable from the files. Within every date, both the CHIRPS and ERA5
extracts list districts in **exactly the same sequence**:

```
SIRONKO, BULAMBULI, BUKWO, BUTALEJA, KWEEN, KAPCHORWA, MANAFWA, BUDUDA, MBALE, MBALE
```

That ordering is identical on every date in both datasets. Since the CHIRPS
notebook *proved* by containment that the first MBALE row is the general district,
the same polygon occupies the same position here. The identification therefore
transfers rather than being assumed.

The cell below verifies the shared ordering rather than trusting it, and fails
loudly if a future re-extraction changes it.

### It also barely matters

Unlike rainfall, where the two MBALE series differed by 1.08 mm on an average day,
the two temperature series differ by **0.095 °C** and correlate at 0.994.
ERA5-Land's ~9 km grid barely resolves a polygon the size of Mbale Municipality,
and temperature is far more spatially smooth than convective rainfall. Picking the
wrong series would be an error of under a tenth of a degree — worth getting right,
but not a threat to the analysis.

In [ ]:
def district_order(frame: pd.DataFrame) -> tuple:
    """The single within-date district sequence, or raise if it is not constant."""
    sequences = (
        frame.sort_values(["source_file", "file_row"])
        .groupby("date")["district"]
        .apply(tuple)
        .unique()
    )
    if len(sequences) != 1:
        raise ValueError(f"within-date district order is not constant: {len(sequences)} variants")
    return sequences[0]


era5_order = district_order(temperature)
chirps_order = district_order(load_extracts(CHIRPS_DIR, "chirps_daily_districts_*.csv"))

print(f"\nERA5   order: {era5_order}")
print(f"CHIRPS order: {chirps_order}")
if era5_order != chirps_order:
    raise ValueError(
        "the two extractions no longer share a district order, so the CHIRPS "
        "identification of the Mbale District polygon cannot be transferred here"
    )

# Established by containment in chirps_preprocessing.ipynb: the FIRST MBALE row
# in the shared ordering is the general district
DISTRICT_POSITION = 0

mbale = temperature[temperature["district"] == "MBALE"].sort_values(
    ["date", "source_file", "file_row"]
).copy()
mbale["position"] = mbale.groupby("date").cumcount()
if set(mbale["position"].unique()) != {0, 1}:
    raise ValueError(f"expected 2 MBALE rows per date, saw {sorted(mbale['position'].unique())}")

print(f"\nshared ordering confirmed; MBALE occupies positions "
      f"{[i for i, d in enumerate(era5_order) if d == 'MBALE']}")
print(f"-> position {DISTRICT_POSITION} is the general Mbale District")

first = mbale[mbale["position"] == 0].set_index("date")
second = mbale[mbale["position"] == 1].set_index("date")
print(f"\nhow much the two series differ:")
print(f"  correlation (tmean):  {first['tmean_c'].corr(second['tmean_c']):.4f}")
print(f"  mean |difference|:    {(first['tmean_c'] - second['tmean_c']).abs().mean():.3f} C")
print(f"  largest difference:   {(first['tmean_c'] - second['tmean_c']).abs().max():.3f} C")

## 3. Restrict to the study area and write

Two sets of rows are removed: Mbale Municipality, and the districts outside the
study area. **BUKWO and KWEEN** were dropped during label preprocessing — high on
the massif, where the wet-season hazard is mass movement rather than riverine
flooding, and contributing 1–2 flood district-days between them.

No values are combined, so `tmax_c`, `tmin_c` and `tmean_c` keep their original
meaning.

In [ ]:
municipality_rows = mbale[mbale["position"] != DISTRICT_POSITION]

before = len(temperature)
temperature = temperature.drop(index=municipality_rows.index)
print(f"dropped {len(municipality_rows):,} Mbale Municipality rows: {before:,} -> {len(temperature):,}")

outside = temperature["district"].isin(EXCLUDED_DISTRICTS)
print(f"dropped {int(outside.sum()):,} rows for {', '.join(EXCLUDED_DISTRICTS)}")
temperature = (
    temperature[~outside]
    .drop(columns=["source_file", "file_row"])
    .sort_values(["district", "date"])
    .reset_index(drop=True)
)

expected_days = pd.date_range(temperature["date"].min(), temperature["date"].max(), freq="D")
gaps = {
    name: len(set(expected_days) - set(group["date"]))
    for name, group in temperature.groupby("district")
}

print(f"\nrows: {len(temperature):,}  districts: {temperature['district'].nunique()}  "
      f"days per district expected: {len(expected_days):,}")
print(f"duplicate district-days: {int(temperature.duplicated(['date', 'district']).sum())}")
incomplete = {name: n for name, n in gaps.items() if n}
print(f"districts with missing days: {incomplete or 'none'}")

OUT = REPO_ROOT / TEMPERATURE_PATH
temperature.to_csv(OUT, index=False)
print(f"\nwrote {len(temperature):,} rows to {OUT}")
print(f"  columns: {', '.join(temperature.columns)}")

## 4. Join onto the flood training frame

The rainfall-based training frame is joined with temperature on `date` and
`district`. ERA5-Land starts in 1981 and the training frame in 1998, so the
temperature record fully covers it and the join should lose nothing — which the
cell asserts rather than assumes.

Only the three raw temperature columns are added. Derived features — rolling
means, and especially **departure from the seasonal normal**, which strips the
annual cycle and is the more informative form for a flood model — are left for
feature engineering rather than baked in here.

In [ ]:
training = pd.read_csv(REPO_ROOT / TRAINING_IN, parse_dates=["date"])
print(f"training frame in:  {len(training):,} rows, "
      f"{training['date'].min():%Y-%m-%d} to {training['date'].max():%Y-%m-%d}")

covered = (temperature["date"].min() <= training["date"].min()
           and temperature["date"].max() >= training["date"].max())
print(f"temperature covers the training range: {covered}")

before = len(training)
training = training.merge(
    temperature[["date", "district"] + TEMP_COLUMNS], on=["date", "district"], how="left"
)

if len(training) != before:
    raise ValueError(f"join changed the row count: {before} -> {len(training)}")
unmatched = int(training[TEMP_COLUMNS].isna().any(axis=1).sum())
if unmatched:
    raise ValueError(f"{unmatched} training rows have no temperature")

print(f"training frame out: {len(training):,} rows, {len(training.columns)} columns, "
      f"{unmatched} unmatched")

OUT = REPO_ROOT / TRAINING_OUT
training.to_csv(OUT, index=False)
print(f"\nwrote {OUT}")
print(f"  columns: {', '.join(training.columns)}")
print(f"  positives: {int(training['Flood occurrences'].sum())}  "
      f"negatives: {int((~training['Flood occurrences']).sum()):,}")
print()
print("temperature on flood days against non-flood days:")
print(training.groupby("Flood occurrences")[TEMP_COLUMNS].mean().round(2).to_string())

### Findings — temperature against the label

Mean temperature on flood days and non-flood days:

| | tmax_c | tmin_c | tmean_c |
| --- | --- | --- | --- |
| non-flood | 23.47 | 14.89 | 19.21 |
| **flood** | 23.68 | 15.61 | 19.58 |

**The separation is small — around 0.4 °C in the mean, and 0.7 °C in `tmin_c`.**
For comparison, 15-day antecedent rainfall differs by roughly 44 mm between the
classes (112.8 against 68.4). Temperature is not going to be a strong predictor
on its own.

The `tmin_c` gap being the largest of the three is the interesting part, and it is
consistent with the physics: flood days are cloudy and humid, and cloud cover
suppresses overnight radiative cooling, so night-time minima rise. That makes
`tmin_c` more a *symptom* of the rainy conditions than an independent driver —
useful to a model as a correlate of cloudiness, but not evidence that heat causes
floods.

This is worth stating plainly in the write-up: temperature earns its place as a
covariate that may help the model discriminate at the margins, not as a
mechanism. The earlier decision to keep temperature out of the negative-filtering
rule is reinforced by these numbers — a 0.4 °C difference could not support a
plausibility threshold.

## 5. Result

**`dataset/era5_daily_temperature.csv`** — 115,052 rows: 16,436 consecutive days
(1981-01-01 to 2025-12-31) across the 7 study districts, one row per
district-day, no duplicates and no gaps. Columns `date`, `district`, `tmax_c`,
`tmin_c`, `tmean_c` in °C.

**`dataset/flood_training_data.csv`** — the model-ready frame: 48,605 rows,
1998-01-15 to 2018-06-20, 121 positives against 48,484 negatives (400:1).
Columns:

| Column | Source |
| --- | --- |
| `date`, `district` | keys |
| `rain_mean`, `rain_max`, `rain_min` | CHIRPS v3, mm/day |
| `ante_15d` | 15-day antecedent rainfall, mm — the variable the negative filter acts on |
| `tmax_c`, `tmin_c`, `tmean_c` | ERA5-Land, °C |
| `Flood occurrences` | DesInventar, boolean |

This supersedes `chirps_flood_training_data.csv`, which is left in place as the
rainfall-only intermediate.

Three things to carry forward:

1. **Temperature separates the classes weakly** — 0.4 °C in the mean against
   44 mm in antecedent rainfall. It is a covariate, not a driver, and `tmin_c` is
   most likely acting as a proxy for cloud cover.
2. **Rainfall remains the binding constraint on the study period.** ERA5-Land
   covers 1981–2025; CHIRPS was extracted only from 1998. Re-extracting CHIRPS
   back to 1991 would recover the two discarded pre-1998 flood onsets.
3. **The 400:1 imbalance is untouched by this step**, as expected — adding
   features does not change class prevalence. That is the next problem.